# Nivel 1

## Parte A

### Carregamento

In [111]:
import pandas as pd
import json

with open("../dados/dados_nivel_1.json", "r") as f:
    dados_1 = json.load(f)

taxa_cambio = dados_1["taxa_cambio_usd_brl"]
df = pd.json_normalize(dados_1["operacoes"])
print("Dataframe: ")
df.head(10)

Dataframe: 


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


### Limpeza

Como o json era pequeno, li cada um dos elementos em busca de qualquer ponto "diferente", o que eu relatei a seguir é o que foi encontrado

**1. `OP-0007` duplicada -> removida com `drop_duplicates()`**

São 20 registros para 19 `id` distintos. As duas linhas de `OP-0007` são idênticas nos 9 campos.

O que decide o tratamento aqui é o `id`, não os valores. `id` é a chave primária da operação, então duas linhas com o mesmo `id` são o mesmo evento gravado duas vezes e não duas transações parecidas. Desse modo, a melhor opção é remover esse ponto


**2. `OP-0017` sem data -> mantida, sem imputar e sem dropar**

Uma linha tem `data: null`, com `observacao: "data nao capturada pelo sistema"`.

*Por que não dropar.* A ausência da data é informação, não vazio. Ela registra uma falha de captura na origem, e metadado faltando é por si só um indicador de anomalia que vale acompanhar. Por exemplo, um ponto em que não tem data registrada e que a transação foi feita em espécie é algo a se notar no contexto de PLD

*Por que não imputar.* Qualquer preenchimento (moda, forward fill, data vizinha do mesmo cliente) inventa uma posição no tempo que o dado não tem. 

In [112]:
df_clean = df.drop_duplicates().reset_index(drop=True)
print("Df sem duplicata: ")
df_clean.head(10)

Df sem duplicata: 


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
9,OP-0010,CLI-A-4,2026-03-03,3800,BRL,cartao,pagamento,Alfa Comercio LTDA,


### Normalização -> 1 único ponto que foi necessário converter de USD -> BRL

In [113]:
df_clean["valor"] = df_clean["valor"].astype(float) # necessário converter tudo para float
mask = df_clean["moeda"] == "USD"
df_clean.loc[mask, "valor"] *= taxa_cambio
df_clean.loc[mask, "moeda"] = "BRL"

df_clean

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100.0,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300.0,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800.0,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300.0,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900.0,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000.0,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200.0,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200.0,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100.0,BRL,pix,transferencia_enviada,Zeta Importacao,
9,OP-0010,CLI-A-4,2026-03-03,3800.0,BRL,cartao,pagamento,Alfa Comercio LTDA,


### Agregações

volume total transacionado por cliente

In [114]:
vol_total_cliente = df_clean.groupby("cliente_id").agg(
    volume_transacionado=("valor", "sum")
)
vol_total_cliente

,volume_transacionado
cliente_id,
CLI-A-1,57500.0
CLI-A-2,52900.0
CLI-A-3,48500.0
CLI-A-4,79500.0
CLI-A-5,16900.0
CLI-A-6,10200.0


quantidade de operações por canal

In [115]:
qnt_op_canal = df_clean.groupby("canal").agg(
    operacoes_canal=("canal", "count")
)
qnt_op_canal

,operacoes_canal
canal,
boleto,3
cartao,2
especie,1
pix,8
ted,5


### Regras determinísticas

Regra 1 - Fracionamento. Sinalize o cliente que, em uma mesma data, realizou 3 ou mais operações cuja soma ultrapassa R$ 50.000,00, sendo que nenhuma operação isolada atinge R$ 20.000,00.

Flags por grupo

In [116]:
lim_soma = 50000      # "soma ultrapassa R$ 50.000,00"
lim_op = 20000  # "nenhuma operação isolada atinge R$ 20.000,00"
min_op = 3

transacoes_cliente_dia = df_clean.groupby(["cliente_id", "data"], dropna=True).agg( # dropna para o caso em que falta data
    n_operacoes=("valor", "count"),
    soma_dia=("valor", "sum"),
    maior_operacao=("valor", "max"),
)

transacoes_cliente_dia["flag_fracionamento"] = (
    (transacoes_cliente_dia["n_operacoes"] >= min_op)
    & (transacoes_cliente_dia["soma_dia"] > lim_soma)
    & (transacoes_cliente_dia["maior_operacao"] < lim_op)
)
transacoes_cliente_dia

n_operacoes  soma_dia  maior_operacao  \
cliente_id data                                                
CLI-A-1    2026-03-09            3   54200.0         18800.0   
           2026-03-21            1    3300.0          3300.0   
CLI-A-2    2026-03-14            2   52900.0         27000.0   
CLI-A-3    2026-03-05            3   48500.0         17200.0   
CLI-A-4    2026-03-03            1    3800.0          3800.0   
           2026-03-11            1    5100.0          5100.0   
           2026-03-18            1    5800.0          5800.0   
           2026-03-24            1   64800.0         64800.0   
CLI-A-5    2026-03-07            1    2900.0          2900.0   
           2026-03-16            1    7000.0          7000.0   
           2026-03-26            1    2700.0          2700.0   
CLI-A-6    2026-03-12            1    8800.0          8800.0   
           2026-03-28            1    1400.0          1400.0   

                       flag_fracionamento  
cliente_id data                            
CLI-A-1    2026-03-09                True  
           2026-03-21               False  
CLI-A-2    2026-03-14               False  
CLI-A-3    2026-03-05               False  
CLI-A-4    2026-03-03               False  
           2026-03-11               False  
           2026-03-18               False  
           2026-03-24               False  
CLI-A-5    2026-03-07               False  
           2026-03-16               False  
           2026-03-26               False  
CLI-A-6    2026-03-12               False  
           2026-03-28               False

Adicionando flag ao *cliente*

In [117]:
df_clean["flag_fracionamento"] = (
    pd.MultiIndex.from_frame(df_clean[["cliente_id", "data"]])
    .map(transacoes_cliente_dia["flag_fracionamento"])
    .fillna(False)  # operação sem data não tem par correspondente -> não sinalizada
    .astype(bool)
)

df_clean["flag_regra1_fracionamento"] = (
    df_clean.groupby("cliente_id")["flag_fracionamento"].transform("any")
)

df_clean = df_clean.drop(columns="flag_fracionamento")
df_clean[["id", "cliente_id", "data", "valor", "flag_regra1_fracionamento"]]

,id,cliente_id,data,valor,flag_regra1_fracionamento
0,OP-0001,CLI-A-1,2026-03-09,18100.0,True
1,OP-0002,CLI-A-1,2026-03-09,17300.0,True
2,OP-0003,CLI-A-1,2026-03-09,18800.0,True
3,OP-0004,CLI-A-1,2026-03-21,3300.0,True
4,OP-0005,CLI-A-2,2026-03-14,25900.0,False
5,OP-0006,CLI-A-2,2026-03-14,27000.0,False
6,OP-0007,CLI-A-3,2026-03-05,17200.0,False
7,OP-0008,CLI-A-3,2026-03-05,15200.0,False
8,OP-0009,CLI-A-3,2026-03-05,16100.0,False
9,OP-0010,CLI-A-4,2026-03-03,3800.0,False


Regra 2 - Valor atípico. Sinalize a operação cujo valor em BRL seja superior a 5× a mediana dos valores daquele mesmo cliente. Aplique apenas a clientes com 4 ou mais operações. 

In [118]:
lim_mediana = 5 
min_op_r2 = 4

transacoes_cliente_mediana = df_clean.groupby(["cliente_id"]).agg(
    n_operacoes=("valor", "count"),
    mediana=("valor", "median"),
)
transacoes_cliente_mediana

,n_operacoes,mediana
cliente_id,,
CLI-A-1,4,17700.0
CLI-A-2,2,26450.0
CLI-A-3,3,16100.0
CLI-A-4,4,5450.0
CLI-A-5,4,3600.0
CLI-A-6,2,5100.0


In [119]:
mediana_cliente = df_clean["cliente_id"].map(transacoes_cliente_mediana["mediana"])
n_op_cliente = df_clean["cliente_id"].map(transacoes_cliente_mediana["n_operacoes"])

df_clean["flag_regra2_valor_atipico"] = (
    (n_op_cliente >= min_op_r2)
    & (df_clean["valor"] > lim_mediana * mediana_cliente)
)

df_clean[["id", "cliente_id", "valor", "flag_regra1_fracionamento", "flag_regra2_valor_atipico"]]

,id,cliente_id,valor,flag_regra1_fracionamento,flag_regra2_valor_atipico
0,OP-0001,CLI-A-1,18100.0,True,False
1,OP-0002,CLI-A-1,17300.0,True,False
2,OP-0003,CLI-A-1,18800.0,True,False
3,OP-0004,CLI-A-1,3300.0,True,False
4,OP-0005,CLI-A-2,25900.0,False,False
5,OP-0006,CLI-A-2,27000.0,False,False
6,OP-0007,CLI-A-3,17200.0,False,False
7,OP-0008,CLI-A-3,15200.0,False,False
8,OP-0009,CLI-A-3,16100.0,False,False
9,OP-0010,CLI-A-4,3800.0,False,False


### Validando as regras

Uma coluna por condição. A regra dispara onde as três dão `True`.

- **Captura** `CLI-A-1 / 09-03`: 3 ops, R$ 54.200, maior R$ 18.800.
- **Não captura** `CLI-A-3 / 05-03`: mesmo desenho, mas soma R$ 48.500 — falha só em `c2`, por R$ 1.500.
- **Não captura** `CLI-A-2 / 14-03`: soma passa de 50k, mas são 2 ops e ambas acima de 20k — falha em `c1` e `c3`.

`CLI-A-3` e `CLI-A-2` são parecidos. No primeiro, sem a limpeza ele viraria falso positivo: com a duplicata `OP-0007`, seriam 4 ops somando R$ 65.700. No segundo, passa dos 50k, mas tem apenas 2 operações

In [120]:
validacao_r1 = transacoes_cliente_dia[transacoes_cliente_dia["n_operacoes"] >= 2].copy()
validacao_r1["c1_3_ou_mais_ops"] = validacao_r1["n_operacoes"] >= min_op
validacao_r1["c2_soma_acima_50k"] = validacao_r1["soma_dia"] > lim_soma
validacao_r1["c3_nenhuma_atinge_20k"] = validacao_r1["maior_operacao"] < lim_op
validacao_r1

,,n_operacoes,soma_dia,maior_operacao,flag_fracionamento,c1_3_ou_mais_ops,c2_soma_acima_50k,c3_nenhuma_atinge_20k
cliente_id,data,,,,,,,
CLI-A-1,2026-03-09,3,54200.0,18800.0,True,True,True,True
CLI-A-2,2026-03-14,2,52900.0,27000.0,False,False,True,False
CLI-A-3,2026-03-05,3,48500.0,17200.0,False,True,False,True


Uma coluna por condição, mais a razão `valor / mediana do cliente`.

- **Captura** `OP-0013`: 11,89× a mediana do CLI-A-4.
- **Não captura** `OP-0015`: 1,94×, a maior entre as elegíveis.
- **Não captura** `OP-0018`: 1,73× e cliente com 2 ops — barrado por `c1`.

In [121]:
validacao_r2 = df_clean[["id", "cliente_id", "valor"]].copy()
validacao_r2["mediana_cliente"] = mediana_cliente
validacao_r2["razao"] = (df_clean["valor"] / mediana_cliente).round(2)
validacao_r2["c1_cliente_com_4_ops"] = n_op_cliente >= min_op_r2
validacao_r2["c2_acima_de_5x"] = df_clean["valor"] > lim_mediana * mediana_cliente

validacao_r2.sort_values("razao", ascending=False).head(5)

,id,cliente_id,valor,mediana_cliente,razao,c1_cliente_com_4_ops,c2_acima_de_5x
12,OP-0013,CLI-A-4,64800.0,5450.0,11.89,True,True
14,OP-0015,CLI-A-5,7000.0,3600.0,1.94,True,False
17,OP-0018,CLI-A-6,8800.0,5100.0,1.73,False,False
16,OP-0017,CLI-A-5,4300.0,3600.0,1.19,True,False
6,OP-0007,CLI-A-3,17200.0,16100.0,1.07,False,False
